# 12 · Score, the Itô generator & Fokker–Planck

`omnibias-score` composes the closed-form field gradient / Hessian into the
machinery of stochastic processes — **no new low-level kernels**:

- score `∇ log p = ∇p / p`,
- the Itô generator `ℒf = b·∇f + ½ tr(a ∇²f)`,
- the Fokker–Planck adjoint `ℒ*p = −∇·(b p) + ½ ∂ᵢ∂ⱼ(aᵢⱼ p)`.

We validate on the 1-D Ornstein–Uhlenbeck process `dX = −θX dt + σ dW`, whose
stationary density is `p∞ = 𝒩(0, σ²/2θ)`. The exact stationary density must
satisfy `ℒ*p∞ = 0`.

In [ ]:
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, PRIMARY, ACCENT, GOOD
from _fields import make_field, Gauss, Poly
set_style()

torch.set_default_dtype(torch.float64)

from omnibias.fields.torch import _ops_dispatch as dispatch
from omnibias.fields.torch.ops.basic import value
from omnibias.score.torch.ops.sde import score, ito_generator, fokker_planck

theta, sigma2 = 1.0, 0.5            # drift rate; diffusion a = sigma^2
var_inf = sigma2 / (2 * theta)      # stationary variance
print("stationary variance =", var_inf)

## The stationary density satisfies `ℒ*p∞ = 0`

`p∞(x) ∝ exp(−θ x² / σ²)` — a Gaussian with the variance above. We build the
exact field `p∞`, supply the OU drift `b(x) = −θx` (with `∇·b = −θ`) and constant
diffusion `a = σ²`, then evaluate the Fokker–Planck adjoint; it should be ~0.

In [ ]:
c = theta / sigma2                          # p_inf proportional to exp(-c x^2)
pfield = make_field(("x",), {"p": (Gauss(c, xp=torch),)}, dispatch)

xs = torch.linspace(-2.5, 2.5, 200).unsqueeze(-1)
state = pfield(xs)

b = -theta * xs                             # drift, shape (B, 1)
a = torch.tensor([[sigma2]])               # diffusion, shape (1, 1)
div_b = torch.full((xs.shape[0],), -theta)  # div b = -theta

Lstar = fokker_planck(state, "p", drift=b, diffusion=a, drift_divergence=div_b)
print("max |L* p_inf| =", float(Lstar.abs().max()))

## Score and the generator

The score `∇ log p∞ = −2θx/σ²` is linear in `x`. The generator `ℒ` applied to the
test function `f(x) = x` returns the drift `b(x) = −θx` (since `∇x = 1`,
`∇²x = 0`).

In [ ]:
sc = score(state, "p")                      # (B, 1)
sc_exact = -2 * c * xs                       # = -2 theta x / sigma^2
print("score max error           =", float((sc - sc_exact).abs().max()))

ffield = make_field(("x",), {"f": (Poly((0.0, 1.0)),)}, dispatch)  # f(x) = x
state_f = ffield(xs)
Lf = ito_generator(state_f, "f", drift=b, diffusion=a)            # should equal b
print("generator L[x] vs drift   =", float((Lf - b.squeeze(-1)).abs().max()))

In [ ]:
xg = xs.squeeze(-1).numpy()
p = value(state, "p").detach().numpy()
dx = xg[1] - xg[0]
p_norm = p / (p.sum() * dx)

fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.2))

axl.plot(xg, p_norm, color=PRIMARY, label=r"$p_\infty \propto e^{-\theta x^2/\sigma^2}$")
axl.fill_between(xg, p_norm, color=PRIMARY, alpha=0.12)
axl.set_xlabel("x")
axl.set_title("OU stationary density")
axl.legend()

axr.plot(xg, Lstar.detach().numpy(), color=ACCENT, label=r"$\mathcal{L}^* p_\infty$ ($\approx 0$)")
axr.plot(xg, sc.squeeze(-1).detach().numpy(), color=GOOD, label=r"score $\nabla\log p_\infty$")
axr.axhline(0.0, color="0.6", lw=1)
axr.set_xlabel("x")
axr.set_title("Fokker–Planck residual & score")
axr.legend()

plt.tight_layout()
plt.show()

## Takeaway

Score, the Itô generator, and the Fokker–Planck adjoint are *compositions* of
omnibias's closed-form gradient and Hessian. The exact OU stationary density
drives the adjoint to ~0 and the score matches `−2θx/σ²` to machine precision —
the calculus backbone for score-based diffusion and SDE models.

Back to the [gallery index](README.md).